In [1]:
import numpy as np
import pandas as pd
from sklearn import preprocessing
import lightgbm as lgb

BASE_PATH = "/kaggle/data/champs-scalar-coupling"

train = pd.read_csv(f"{BASE_PATH}/train.csv")
test = pd.read_csv(f"{BASE_PATH}/test.csv")
sub = pd.read_csv(f"{BASE_PATH}/sample_submission.csv")
print(train.shape, test.shape, sub.shape)

for i in range(4):
    lbl = preprocessing.LabelEncoder()
    all_chars = pd.concat([train["type"], test["type"]], axis=0).map(
        lambda x: str(x)[i]
    )
    lbl.fit(all_chars.values)
    train["type" + str(i)] = lbl.transform(train["type"].map(lambda x: str(x)[i]))
    test["type" + str(i)] = lbl.transform(test["type"].map(lambda x: str(x)[i]))

structures = pd.read_csv(f"{BASE_PATH}/structures.csv")

s0 = structures.rename(
    columns={
        "atom_index": "atom_index_0",
        "atom": "atom0",
        "x": "x0",
        "y": "y0",
        "z": "z0",
    }
)
train = pd.merge(
    train,
    s0[["molecule_name", "atom_index_0", "atom0", "x0", "y0", "z0"]],
    how="left",
    on=["molecule_name", "atom_index_0"],
)
test = pd.merge(
    test,
    s0[["molecule_name", "atom_index_0", "atom0", "x0", "y0", "z0"]],
    how="left",
    on=["molecule_name", "atom_index_0"],
)

s1 = structures.rename(
    columns={
        "atom_index": "atom_index_1",
        "atom": "atom1",
        "x": "x1",
        "y": "y1",
        "z": "z1",
    }
)
train = pd.merge(
    train,
    s1[["molecule_name", "atom_index_1", "atom1", "x1", "y1", "z1"]],
    how="left",
    on=["molecule_name", "atom_index_1"],
)
test = pd.merge(
    test,
    s1[["molecule_name", "atom_index_1", "atom1", "x1", "y1", "z1"]],
    how="left",
    on=["molecule_name", "atom_index_1"],
)

del structures, s0, s1
print(train.shape, test.shape, sub.shape)



(4191263, 6) (467813, 5) (467813, 2)
(4191263, 18) (467813, 17) (467813, 2)


In [2]:
train_p0 = train[["x0", "y0", "z0"]].fillna(0).values
train_p1 = train[["x1", "y1", "z1"]].fillna(0).values
test_p0 = test[["x0", "y0", "z0"]].fillna(0).values
test_p1 = test[["x1", "y1", "z1"]].fillna(0).values

train["dist"] = np.linalg.norm(train_p0 - train_p1, axis=1)
test["dist"] = np.linalg.norm(test_p0 - test_p1, axis=1)



In [3]:
for c in ["atom0", "atom1"]:
    le = preprocessing.LabelEncoder()
    all_vals = pd.concat([train[c], test[c]], axis=0).astype(str).fillna("nan")
    le.fit(all_vals.values)
    train[c] = le.transform(train[c].astype(str).fillna("nan"))
    test[c] = le.transform(test[c].astype(str).fillna("nan"))

col = [
    c
    for c in train.columns
    if c not in ["id", "molecule_name", "scalar_coupling_constant", "type"]
]

drop_const_type_feats = {"type0", "type1", "type2", "type3"}
col = [c for c in col if c not in drop_const_type_feats]

params = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "mae",
    # Change (score-relevant): remove per-type LR/round scaling by using one stable LR,
    # which is typically better calibrated across coupling types for this competition.
    "learning_rate": 0.2,
    "num_leaves": 64,
    "seed": 99,
    "feature_fraction_seed": 99,
    "bagging_seed": 99,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "min_data_in_leaf": 50,
    "min_sum_hessian_in_leaf": 1e-3,
    "lambda_l2": 2.0,
    "min_gain_to_split": 1e-3,
}

rng = np.random.RandomState(99)
mols = train["molecule_name"].unique()
rng.shuffle(mols)
cut = int(len(mols) * 0.8)
train_mols = set(mols[:cut])
valid_mols = set(mols[cut:])

trn_idx = train["molecule_name"].isin(train_mols)
val_idx = train["molecule_name"].isin(valid_mols)

type_mean_trn = train.loc[trn_idx].groupby("type")["dist"].mean()

type_mean_full = type_mean_trn.copy()
type_mean_trn_safe = type_mean_trn

mol_type_mean_trn = train.loc[trn_idx].groupby(["molecule_name", "type"])["dist"].mean()


def compute_dist_to_mean(df, mol_type_mean_series, type_mean_series):
    keys = pd.MultiIndex.from_frame(df[["molecule_name", "type"]])
    mol_type_mean = mol_type_mean_series.reindex(keys).to_numpy()
    type_mean = df["type"].map(type_mean_series).to_numpy()
    den = np.where(np.isfinite(mol_type_mean), mol_type_mean, type_mean)
    out = df["dist"].to_numpy() / den
    out = np.where(np.isfinite(out), out, 0.0)
    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
    return out


train.loc[trn_idx, "dist_to_type_mean"] = compute_dist_to_mean(
    train.loc[trn_idx, :], mol_type_mean_trn, type_mean_trn_safe
)
train.loc[val_idx, "dist_to_type_mean"] = compute_dist_to_mean(
    train.loc[val_idx, :], mol_type_mean_trn, type_mean_trn_safe
)
test["dist_to_type_mean"] = compute_dist_to_mean(
    test, mol_type_mean_trn, type_mean_trn_safe
)

train["dist_to_type_mean"] = train["dist_to_type_mean"].astype(np.float32).fillna(0.0)
test["dist_to_type_mean"] = test["dist_to_type_mean"].astype(np.float32).fillna(0.0)

train["dist_to_type_mean_full"] = (
    (train["dist"] / train["type"].map(type_mean_full))
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
    .astype(np.float32)
)
test["dist_to_type_mean_full"] = (
    (test["dist"] / test["type"].map(type_mean_full))
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
    .astype(np.float32)
)

x_test_full = test[col].copy()
for c in x_test_full.columns:
    if not (
        np.issubdtype(x_test_full[c].dtype, np.number) or x_test_full[c].dtype == bool
    ):
        x_test_full[c] = pd.to_numeric(x_test_full[c], errors="coerce")
x_test_full.fillna(0.0, inplace=True)

if ("dist_to_type_mean" in x_test_full.columns) and (
    "dist_to_type_mean_full" in test.columns
):
    x_test_full["dist_to_type_mean"] = test["dist_to_type_mean_full"].values

test_pred = np.zeros(len(test), dtype=np.float64)

types = train["type"].unique()
print("Training per type models:", len(types))

# Change (score-relevant): use one consistent boosting schedule for every type, and let
# early stopping pick best_iteration. This avoids type-size-based capacity misallocation.
num_boost_round = 2000
early_stop_rounds = 50

for t in sorted(types):
    trn_mask_t = trn_idx & (train["type"] == t)
    val_mask_t = val_idx & (train["type"] == t)
    tst_mask_t = test["type"] == t

    if tst_mask_t.sum() == 0:
        continue

    x1 = train.loc[trn_mask_t, col].copy()
    x2 = train.loc[val_mask_t, col].copy()
    y1 = train.loc[trn_mask_t, "scalar_coupling_constant"]
    y2 = train.loc[val_mask_t, "scalar_coupling_constant"]

    for df in (x1, x2):
        for cc in df.columns:
            if not (np.issubdtype(df[cc].dtype, np.number) or df[cc].dtype == bool):
                df[cc] = pd.to_numeric(df[cc], errors="coerce")
        df.fillna(0.0, inplace=True)

    dtrain = lgb.Dataset(x1, label=y1)
    dvalid = lgb.Dataset(x2, label=y2, reference=dtrain) if len(x2) > 0 else None

    valid_sets = [dvalid] if dvalid is not None else []
    valid_names = ["valid"] if dvalid is not None else []

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        valid_sets=valid_sets,
        valid_names=valid_names,
        feval=None,
        callbacks=[
            (
                lgb.early_stopping(
                    stopping_rounds=early_stop_rounds,
                    first_metric_only=True,
                    verbose=True,
                )
                if dvalid is not None
                else lgb.log_evaluation(period=0)
            ),
            lgb.log_evaluation(period=50),
        ],
    )

    best_iter = getattr(model, "best_iteration", None) or num_boost_round

    trn_all_mask_t = train["type"] == t
    x_all = train.loc[trn_all_mask_t, col].copy()
    if "dist_to_type_mean" in x_all.columns:
        x_all["dist_to_type_mean"] = train.loc[
            trn_all_mask_t, "dist_to_type_mean_full"
        ].values
    y_all = train.loc[trn_all_mask_t, "scalar_coupling_constant"]

    for cc in x_all.columns:
        if not (np.issubdtype(x_all[cc].dtype, np.number) or x_all[cc].dtype == bool):
            x_all[cc] = pd.to_numeric(x_all[cc], errors="coerce")
    x_all.fillna(0.0, inplace=True)

    dtrain_all = lgb.Dataset(x_all, label=y_all)
    model_all = lgb.train(
        params,
        dtrain_all,
        num_boost_round=int(best_iter),
        valid_sets=[],
        valid_names=[],
        feval=None,
        callbacks=[lgb.log_evaluation(period=0)],
    )

    test_pred[tst_mask_t.values] = model_all.predict(
        x_test_full.loc[tst_mask_t.values, :],
        num_iteration=int(best_iter),
    )

test["scalar_coupling_constant"] = test_pred

submission = test[["id", "scalar_coupling_constant"]].copy()
submission = submission.sort_values("id").reset_index(drop=True)
submission.to_csv("submission.csv", float_format="%.9f", index=False)
print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)

Training per type models: 8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013146 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1824
[LightGBM] [Info] Number of data points in the train set: 510053, number of used features: 9
[LightGBM] [Info] Start training from score 94.982104
Training until validation scores don't improve for 50 rounds
[50]	valid's l1: 4.45685
[100]	valid's l1: 4.33258
[150]	valid's l1: 4.25199
[200]	valid's l1: 4.19528
[250]	valid's l1: 4.15766
[300]	valid's l1: 4.11471
[350]	valid's l1: 4.08044
[400]	valid's l1: 4.0503
[450]	valid's l1: 4.02929
[500]	valid's l1: 4.00633
[550]	valid's l1: 3.98817
[600]	valid's l1: 3.97317
[650]	valid's l1: 3.95768
[700]	valid's l1: 3.94491
[750]	valid's l1: 3.9312
[800]	valid's l1: 3.91602
[850]	valid's l1: 3.90517
[900]	valid's l1: 3.89448
[950]	valid's l1: 3.88538
[1000]	valid's l1: 3.87614
[1050]	valid's l1: 3.86671
[1100]	valid's l1: 3.85